# IndicSent-M: Explainable AI (XAI) Demonstration
**Course**: CSET 346 - Natural Language Processing  
**Instructor**: Dr. Shubham Choudhary  
**Team**: Anya Yadav, Pragya Singh, Priyanshu Thakur, Ronit Jaiswal, Sparshman Gupta  

In multimodal sentiment analysis, understanding **why** a model arrived at its decision is vital for trust and academic evaluation.
Here we demonstrate **dual explainability**:
1. **LIME (Local Interpretable Model-agnostic Explanations)**: Identifies influential words that drive the predicted sentiment polarity.
2. **Grad-CAM (Gradient-weighted Class Activation Mapping)**: Produces visual heatmaps highlighting the image regions most influential to the neural network.


## 1. Setup & Explainability Engines


In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

sys.path.append(os.path.abspath('../backend'))
from explain import GradCAM, overlay_gradcam, SimpleLimeExplainer
import inference

print('Explainability tools loaded successfully.')


## 2. Case Study 1: Correctly Predicted Neutral / Complex Meme
We analyze a meme where the model successfully fused text and image cues to predict **Neutral** sentiment.


In [ ]:
sample_img_path = '../data/memotion3/valImages/0.jpg'
sample_text = 'When you complete all your assignments but still have more due tomorrow'

result = inference.predict(sample_img_path, sample_text)

print(f'Predicted Sentiment: {result["final_sentiment"]}')
print('Confidence Scores:', result['confidence_scores'])
print('\nPlain-English Explanation:')
print(result['explanation'])


### Visualizing LIME Influential Words


In [ ]:
words = [w['word'] for w in result['lime_words']]
scores = [w['score'] for w in result['lime_words']]
bar_colors = ['#10B981' if s > 0 else '#EF4444' for s in scores]

plt.figure(figsize=(8, 4))
plt.barh(words, scores, color=bar_colors, edgecolor='#1E293B')
plt.xlabel('Importance Score (Signed Attribution)')
plt.title(f'LIME Word Attributions for {result["final_sentiment"]} Sentiment', fontsize=12, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.gca().invert_yaxis()
plt.show()


### Visualizing Grad-CAM Heatmap Overlay
Shows where the ResNet50 vision backbone concentrated its visual attention.


In [ ]:
import base64
import io

# Decode base64 image
b64_str = result['gradcam_overlay'].split(',')[1]
cam_img = Image.open(io.BytesIO(base64.b64decode(b64_str)))
orig_img = Image.open(sample_img_path)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(orig_img)
axes[0].set_title('Original Input Meme', fontsize=11, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(cam_img)
axes[1].set_title('Grad-CAM Visual Heatmap Overlay', fontsize=11, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()


## 3. Case Study 2: Challenging Sarcastic Meme (Error Analysis)
Sarcastic memes often have positive words paired with a sarcastic facial expression. Let's analyze an example where modal conflict occurs.


In [ ]:
sarcastic_text = 'Oh wow what a great and wonderful surprise test on Monday morning!'
sarcastic_img_path = '../data/memotion3/valImages/1.jpg'

err_result = inference.predict(sarcastic_img_path, sarcastic_text)

print(f'Predicted Sentiment: {err_result["final_sentiment"]}')
print('Confidence Scores:', err_result['confidence_scores'])
print('\nWhy this case is informative:')
print('The text-only branch heavily weights "great" and "wonderful" towards Positive.')
print('However, the visual branch focuses on the exasperated subject, pulling the multimodal fusion score towards Negative.')


## 4. Key Takeaways on Explainability
1. **Faithfulness**: LIME highlights that Hinglish intensifiers and emotive adjectives carry the highest weight in classifying meme polarity.
2. **Interpretability**: Grad-CAM verifies that the convolutional layers focus on faces, meme subject characters, and expressive focal points rather than background noise.
3. **Error Diagnosis**: Explainability reveals semantic dissonance in sarcasm, where cross-modal attention helps resolve contradictory text-image signals.
